In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge, ElasticNet
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import xgboost as xgb
import lightgbm as lgb

In [2]:
DATA_DIR = "/data1/yashvi_bhuva/BP_estimation_using_PPG/UCI/Feature_extraction/Non_fiducial_features/8sec_50%overlap/data"

train_path = os.path.join(DATA_DIR, "SBP_MRMR_train.csv")
val_path   = os.path.join(DATA_DIR, "SBP_MRMR_val.csv")
test_path  = os.path.join(DATA_DIR, "SBP_MRMR_test.csv")

df_train = pd.read_csv(train_path)
df_val   = pd.read_csv(val_path)
df_test  = pd.read_csv(test_path)

print("Train:", df_train.shape)
print("Val:  ", df_val.shape)
print("Test: ", df_test.shape)

Train: (522837, 16)
Val:   (66013, 16)
Test:  (66296, 16)


In [3]:
print(df_train.head()) 
print(df_train.columns.tolist())

   apg_energy_variance  ppg_zero_crossing_rate  ppg_skewness  vpg_skewness  \
0            -0.016488                     0.0     -2.663371     -0.486552   
1            -0.016400                     0.0     -2.427994     -0.587361   
2            -0.016504                     0.0     -2.268329     -0.515821   
3            -0.013686                     0.0     -2.356231     -0.278815   
4            -0.016998                     0.0     -2.517468     -0.572304   

   ppg_energy_variance  apg_mean  apg_energy_kurtosis  apg_zero_crossing_rate  \
0             0.395597 -1.512525            -0.096641               -0.315485   
1             0.448589 -0.134683            -0.159527                0.754060   
2             0.356869  1.426391            -0.166417               -0.009901   
3             0.205878  1.326248             3.040485               -0.544674   
4             0.204584 -0.022311            -0.143879               -0.315485   

   vpg_mean  apg_median  ppg_energy_kurtosis

In [4]:
TARGET = "SBP"

X_train = df_train.drop(columns=[TARGET])
y_train = df_train[TARGET]

X_val = df_val.drop(columns=[TARGET])
y_val = df_val[TARGET]

X_test = df_test.drop(columns=[TARGET])
y_test = df_test[TARGET]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:  ", X_val.shape)
print("y_val:  ", y_val.shape)
print("X_test: ", X_test.shape)
print("y_test: ", y_test.shape)

X_train: (522837, 15)
y_train: (522837,)
X_val:   (66013, 15)
y_val:   (66013,)
X_test:  (66296, 15)
y_test:  (66296,)


In [5]:
X_dev = pd.concat([X_train, X_val], axis=0)
y_dev = pd.concat([y_train, y_val], axis=0)

In [6]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def print_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    print(f"RMSE: {rmse:.4f}")
    print(f"MSE : {mse:.4f}")
    print(f"MAE : {mae:.4f}")
    print(f"R²  : {r2:.4f}")

## SIMPLE LINEAR REGRESSION ##

In [7]:
from sklearn.linear_model import LinearRegression

linear_model = LinearRegression()

linear_model.fit(X_dev, y_dev)

y_test_pred = linear_model.predict(X_test)

print("Simple Linear Regression")
print("------------------------")
print_metrics(y_test, y_test_pred)

Simple Linear Regression
------------------------
RMSE: 22.1539
MSE : 490.7954
MAE : 17.7597
R²  : 0.0768


## INTERACTION LINEAR REGRESSION ##

In [8]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

interaction_model = Pipeline([
    ("poly", PolynomialFeatures(
        degree=2,
        interaction_only=True,
        include_bias=False
    )),
    ("linear", LinearRegression())
])

param_grid = {
    "linear__fit_intercept": [True, False]
}

grid_interaction = GridSearchCV(
    interaction_model,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=3,
    n_jobs=2
)

grid_interaction.fit(X_dev, y_dev)

print("Best parameters:")
print(grid_interaction.best_params_)

y_test_pred = grid_interaction.predict(X_test)

print("\nInteraction Linear Regression")
print("-----------------------------")
print_metrics(y_test, y_test_pred)

Best parameters:
{'linear__fit_intercept': True}

Interaction Linear Regression
-----------------------------
RMSE: 243.2555
MSE : 59173.2335
MAE : 18.3579
R²  : -110.3082


## Robust Linear Regression ##

In [9]:
from sklearn.linear_model import HuberRegressor

robust_model = HuberRegressor(
    max_iter=1000
)

param_grid = {
    "epsilon": [1.1, 1.35, 1.5, 2.0],
    "alpha": [0.0001, 0.001, 0.01, 0.1, 1.0]
}

grid_robust = GridSearchCV(
    robust_model,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=3,
    n_jobs=2
)

grid_robust.fit(X_dev, y_dev)

print("Best parameters:")
print(grid_robust.best_params_)

y_test_pred = grid_robust.predict(X_test)

print("\nRobust Linear Regression")
print("------------------------")
print_metrics(y_test, y_test_pred)

Best parameters:
{'alpha': 1.0, 'epsilon': 2.0}

Robust Linear Regression
------------------------
RMSE: 22.1202
MSE : 489.3016
MAE : 17.7452
R²  : 0.0796


## Fine Decision Tree ##

In [10]:
from sklearn.tree import DecisionTreeRegressor

fine_tree = DecisionTreeRegressor(
    random_state=42
)

param_grid = {
    "max_depth": [10, 15, 20, 25, 30, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

grid_fine_tree = GridSearchCV(
    fine_tree,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=3,
    n_jobs=2
)

grid_fine_tree.fit(X_dev, y_dev)

print("Best parameters:")
print(grid_fine_tree.best_params_)

y_test_pred = grid_fine_tree.predict(X_test)

print("\nFine Decision Tree")
print("------------------")
print_metrics(y_test, y_test_pred)

Best parameters:
{'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 2}

Fine Decision Tree
------------------
RMSE: 20.5534
MSE : 422.4425
MAE : 15.9476
R²  : 0.2054


## Random Forest ##

In [11]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    random_state=42,
    n_jobs=2
)

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt"]
}

grid_rf = GridSearchCV(
    rf,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=3,
    n_jobs=2
)

grid_rf.fit(X_dev, y_dev)

print("Best parameters:")
print(grid_rf.best_params_)

y_test_pred = grid_rf.predict(X_test)

print("\nRandom Forest")
print("-------------")
print_metrics(y_test, y_test_pred)

/data1/yashvi_bhuva/BP_estimation_using_PPG/UCI/venv/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best parameters:
{'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}

Random Forest
-------------
RMSE: 17.5874
MSE : 309.3167
MAE : 13.1610
R²  : 0.4182


## Extra TREES ##

In [12]:
from sklearn.ensemble import ExtraTreesRegressor

extra_trees = ExtraTreesRegressor(
    random_state=42,
    n_jobs=2
)

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 20, 30],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": [1.0, "sqrt"]
}

grid_extra_trees = GridSearchCV(
    estimator=extra_trees,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=3,
    n_jobs=2
)

grid_extra_trees.fit(X_dev, y_dev)

print("Best parameters:")
print(grid_extra_trees.best_params_)

y_test_pred = grid_extra_trees.predict(X_test)

print("Extra Trees Regressor")
print("---------------------")
print_metrics(y_test, y_test_pred)

/data1/yashvi_bhuva/BP_estimation_using_PPG/UCI/venv/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best parameters:
{'max_depth': None, 'max_features': 1.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Extra Trees Regressor
---------------------
RMSE: 17.7040
MSE : 313.4331
MAE : 13.1112
R²  : 0.4104


## XGBOOST ##

In [13]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=2
)

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 6],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

grid_xgb = GridSearchCV(
    xgb,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=3,
    n_jobs=2
)

grid_xgb.fit(X_dev, y_dev)

print("Best parameters:")
print(grid_xgb.best_params_)

y_test_pred = grid_xgb.predict(X_test)

print("\nXGBoost")
print("-------")
print_metrics(y_test, y_test_pred)

/data1/yashvi_bhuva/BP_estimation_using_PPG/UCI/venv/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best parameters:
{'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 200, 'subsample': 0.8}

XGBoost
-------
RMSE: 19.0342
MSE : 362.3010
MAE : 14.7674
R²  : 0.3185


## Light GBM ##

In [14]:
from lightgbm import LGBMRegressor
from sklearn.model_selection import GridSearchCV

lgbm = LGBMRegressor(
    objective="regression",
    random_state=42,
    n_jobs=2,
    verbosity=-1
)

param_grid = {
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1],
    "num_leaves": [15, 31],
    "max_depth": [-1, 10]
}

grid_lgbm = GridSearchCV(
    estimator=lgbm,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=3,
    n_jobs=2,
    verbose=2
)

grid_lgbm.fit(X_dev, y_dev)

print("Best parameters:")
print(grid_lgbm.best_params_)

y_test_pred = grid_lgbm.predict(X_test)

print("\nLightGBM")
print("--------")
print_metrics(y_test, y_test_pred)

Fitting 3 folds for each of 16 candidates, totalling 48 fits
[CV] END learning_rate=0.05, max_depth=-1, n_estimators=100, num_leaves=15; total time=   2.0s
[CV] END learning_rate=0.05, max_depth=-1, n_estimators=100, num_leaves=15; total time=   2.0s
[CV] END learning_rate=0.05, max_depth=-1, n_estimators=100, num_leaves=15; total time=   2.0s
[CV] END learning_rate=0.05, max_depth=-1, n_estimators=100, num_leaves=31; total time=   2.6s
[CV] END learning_rate=0.05, max_depth=-1, n_estimators=100, num_leaves=31; total time=   2.4s
[CV] END learning_rate=0.05, max_depth=-1, n_estimators=100, num_leaves=31; total time=   2.4s
[CV] END learning_rate=0.05, max_depth=-1, n_estimators=200, num_leaves=15; total time=   3.5s
[CV] END learning_rate=0.05, max_depth=-1, n_estimators=200, num_leaves=15; total time=   3.4s
[CV] END learning_rate=0.05, max_depth=-1, n_estimators=200, num_leaves=15; total time=   3.4s
[CV] END learning_rate=0.05, max_depth=-1, n_estimators=200, num_leaves=31; total ti

##  Matern 5/2 gpr ##

In [15]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    Matern,
    ConstantKernel,
    WhiteKernel
)
from sklearn.model_selection import GridSearchCV
import numpy as np

# --------------------------------------------------
# Take a manageable subset for GPR
# --------------------------------------------------

N_GPR = 5000

rng = np.random.RandomState(42)

indices = rng.choice(
    len(X_dev),
    size=N_GPR,
    replace=False
)

X_gpr = X_dev.iloc[indices]
y_gpr = y_dev.iloc[indices]


# --------------------------------------------------
# Matern 5/2 GPR
# --------------------------------------------------

gpr_matern = GaussianProcessRegressor(
    normalize_y=True,
    random_state=42,
    n_restarts_optimizer=1
)


# --------------------------------------------------
# Grid of Matern 5/2 kernels
# --------------------------------------------------

param_grid = {
    "kernel": [
        ConstantKernel(1.0)
        * Matern(
            length_scale=0.5,
            nu=2.5
        )
        + WhiteKernel(
            noise_level=1.0
        ),

        ConstantKernel(1.0)
        * Matern(
            length_scale=1.0,
            nu=2.5
        )
        + WhiteKernel(
            noise_level=1.0
        ),

        ConstantKernel(1.0)
        * Matern(
            length_scale=2.0,
            nu=2.5
        )
        + WhiteKernel(
            noise_level=1.0
        )
    ]
}


# --------------------------------------------------
# GridSearchCV
# --------------------------------------------------

grid_gpr_matern = GridSearchCV(
    estimator=gpr_matern,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=3,
    n_jobs=1
)


# --------------------------------------------------
# Fit
# --------------------------------------------------

grid_gpr_matern.fit(
    X_gpr,
    y_gpr
)


# --------------------------------------------------
# Best parameters
# --------------------------------------------------

print("Best parameters:")
print(grid_gpr_matern.best_params_)


# --------------------------------------------------
# Test prediction
# --------------------------------------------------

y_test_pred = grid_gpr_matern.predict(X_test)


# --------------------------------------------------
# Metrics
# --------------------------------------------------

print("\nMatern 5/2 GPR")
print("--------------")

print_metrics(y_test, y_test_pred)

Best parameters:
{'kernel': 1**2 * Matern(length_scale=2, nu=2.5) + WhiteKernel(noise_level=1)}

Matern 5/2 GPR
--------------
RMSE: 20.9702
MSE : 439.7494
MAE : 16.5521
R²  : 0.1728


## Rational Quadratic GPR ##

In [ ]:
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    RationalQuadratic,
    WhiteKernel
)
gpr = GaussianProcessRegressor(
    normalize_y=True,
    random_state=42
)

param_grid = {
    "kernel": [
        ConstantKernel(1.0) *
        RationalQuadratic(
            length_scale=1.0,
            alpha=1.0
        ) +
        WhiteKernel(noise_level=1.0),

        ConstantKernel(1.0) *
        RationalQuadratic(
            length_scale=0.5,
            alpha=1.0
        ) +
        WhiteKernel(noise_level=1.0),

        ConstantKernel(1.0) *
        RationalQuadratic(
            length_scale=2.0,
            alpha=1.0
        ) +
        WhiteKernel(noise_level=1.0)
    ]
}

grid_gpr_rq = GridSearchCV(
    estimator=gpr,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=3,
    n_jobs=1
)

grid_gpr_rq.fit(
    X_gpr,
    y_gpr
)

print("Best parameters:")
print(grid_gpr_rq.best_params_)


y_test_pred = grid_gpr_rq.predict(X_test)


print("Rational Quadratic GPR")
print("----------------------")
print_metrics(y_test, y_test_pred)

/data1/yashvi_bhuva/BP_estimation_using_PPG/UCI/venv/lib/python3.10/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/data1/yashvi_bhuva/BP_estimation_using_PPG/UCI/venv/lib/python3.10/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


## ANN ##

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import GridSearchCV

ann = MLPRegressor(
    random_state=42,
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20
)

param_grid = {
    "hidden_layer_sizes": [
        (64, 32),
        (128, 64),
        (64, 32, 16)
    ],
    "activation": ["relu"],
    "alpha": [0.0001, 0.001, 0.01],
    "learning_rate_init": [0.001, 0.01],
    "batch_size": [256, 512]
}

grid_ann = GridSearchCV(
    estimator=ann,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=3,
    n_jobs=2,
    verbose=1
)

grid_ann.fit(X_gpr, y_gpr)

print("Best parameters:")
print(grid_ann.best_params_)

y_test_pred = grid_ann.predict(X_test)

print("ANN / MLP Regressor")
print("-------------------")
print_metrics(y_test, y_test_pred)